# 02 — Preprocessing & categorical analysis

Mirrors preprocessing + Cramér's V section from the main notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

hour = pd.read_csv("../data/raw/hour.csv", parse_dates=["dteday"])

from scipy.stats.contingency import association

categorical_cols = ['season', 'yr', 'mnth', 'hr',
    'holiday', 'weekday', 'workingday', 'weathersit']

cat_association = pd.DataFrame(index = categorical_cols, columns = categorical_cols)

# Compare each pair of categorical features with others using Cramér's V

for col1 in categorical_cols:
    for col2 in categorical_cols:
        if col1 == col2:
            cat_association.loc[col1, col2] = 1
        else:
            table = pd.crosstab(hour[col1], hour[col2])
            score = association(table, method = 'cramer', correction = False)
            cat_association.loc[col1, col2] = score

cat_association = cat_association.astype(float)
plt.figure(figsize=(10,6))
sns.heatmap(cat_association, annot=True, cmap = 'Blues', fmt = '.2f')
plt.title("Categorical Feature Association Heatmap (Cramer's V)")
plt.show()

In [ ]:
for col in categorical_cols:
    print("\n", col)
    print(hour.groupby(col)['cnt'].mean().sort_values())

In [ ]:
for col in categorical_cols:
    plt.figure(figsize=(6,4))
    sns.barplot(x=col, y='cnt', data=hour)
    plt.title(f"{col} vs cnt")
    plt.show()

In [ ]:
# Importance Score for the categorical features
importance = {}
for col in categorical_cols:
    means = hour.groupby(col)['cnt'].mean()
    importance[col] =  means.max() - means.min()

importance_df = pd.DataFrame.from_dict(importance, orient='index', columns=['Importance'])
importance_df = importance_df.sort_values(by='Importance', ascending=False)
importance_df
importance_df.plot(kind='bar', figsize=(10,5), title="Categorical Feature Importance (Mean Difference)")
plt.show()